In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from env_workers import env_worker
import torch.multiprocessing as mp
from utils import get_model
import json
from types import SimpleNamespace
import numpy as np

In [19]:
with open('config.json', "r") as config_file:
    config_file = json.load(config_file)
    config = SimpleNamespace(**config_file)

In [37]:
EPSILON = 0.1
DEVICE = 'cpu'
NUM_WORKERS = 8
STEPS = 1
GAMMA = 1.0
T_MAX = 20

model = get_model(config.actor)

In [21]:
parent_remote, worker_remote = zip(*[mp.Pipe() for _ in range(NUM_WORKERS)])
processes = [
    mp.Process(target=env_worker, args=(worker_remote[i], parent_remote[i], i, EPSILON, DEVICE))
    for i in range(NUM_WORKERS)
]

In [22]:
for p in processes:
    p.start()

In [23]:
for remote in worker_remote:
    remote.close()

<frozen importlib._bootstrap>:491: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
<frozen importlib._bootstrap>:491: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
<frozen importlib._bootstrap>:491: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation

In [28]:
for remote in parent_remote:
    remote.send(('reset', None))
results = [remote.recv() for remote in parent_remote]
states, action_masks = zip(*results)
states = torch.vstack(states)
action_masks = np.vstack(action_masks)

In [161]:
rewards, state_values, log_probs, entropies, terminates = [], [], [], [], []

for t in range(T_MAX):
    logits, values = model(states)

    # prepare the mask
    action_masks = torch.tensor(action_masks, dtype=torch.bool)
    action_masks = ~action_masks

    masked_logits = torch.masked_fill(logits, action_masks, value=-torch.inf)
    probs = F.softmax(masked_logits, dim=-1)

    dist = Categorical(probs=probs)
    actions = dist.sample()
    log_prob = dist.log_prob(actions)
    entropy = dist.entropy()

    actions = actions.cpu().detach().tolist()

    for remote, action in zip(parent_remote, actions):
        remote.send(('step', action))


    results = [remote.recv() for remote in parent_remote]
    next_states, action_masks, reward, terminated, outcomes  = zip(*results)
    next_states = torch.vstack(next_states)
    action_masks = np.vstack(action_masks)
    reward = np.vstack(reward)
    terminated = np.vstack(terminated)

    rewards.append(reward)
    state_values.append(values)
    log_probs.append(log_prob)
    entropies.append(entropies)
    terminates.append(terminated)

    states = next_states


with torch.no_grad():
    _, next_values = model(next_states)

R = next_values.numpy()
returns = []
for i in reversed(range(len(rewards))):
    r = rewards[i]
    term = terminates[i].astype(np.float32)
    R = r + (1 - term) * GAMMA * R
    returns.insert(0, R)
